# Ajuste de Parámetros

## Un caso aplicado: Asistente de Trámites Académicos

Este notebook desarrolla un ejercicio práctico organizado en tres etapas sucesivas, en el que cada técnica revisada resuelve una limitación identificada en la etapa anterior. El propósito es que la comparación entre etapas, realizada sobre el mismo conjunto de preguntas de evaluación, permita observar de manera directa el alcance y las limitaciones de cada enfoque.

**Caso de estudio.** Se plantea el desarrollo de un asistente conversacional orientado a resolver consultas de estudiantes sobre trámites académicos (cancelación de asignaturas, homologaciones, matrícula y requisitos de grado). El asistente debe cumplir dos condiciones simultáneas:

1. Responder con información correcta y verificable, sustentada en el reglamento institucional vigente.
2. Responder siempre bajo un formato estructurado y un tono institucional definidos previamente.

**Etapas del ejercicio:**

| Etapa | Técnica | Problema que resuelve | Problema que persiste |
|---|---|---|---|
| 1 | Prompt Engineering | Control del tono y del formato de respuesta | El modelo carece de conocimiento del reglamento real y genera información incorrecta |
| 2 | RAG (Retrieval-Augmented Generation) | Incorporación de conocimiento verificable mediante recuperación de documentos | El formato de respuesta se degrada ante preguntas ambiguas o fuera de alcance |
| 3 | LoRA (Low-Rank Adaptation) | Consolidación de un comportamiento y formato consistentes, incluso en casos no vistos durante el entrenamiento | — |

**Documento de referencia.** El ejercicio utiliza un reglamento institucional de carácter ficticio, elaborado exclusivamente con fines académicos, disponible como documento independiente (`reglamento_tramites_academicos.docx`) y también incorporado como texto dentro de este notebook para garantizar su ejecución autónoma.


## Estructura del notebook

- **Sección 0.** Configuración del entorno, carga del modelo base y definición del conjunto de preguntas de evaluación.
- **Sección 1.** Etapa 1 — Prompt Engineering.
- **Sección 2.** Etapa 2 — RAG.
- **Sección 3.** Etapa 3 — LoRA.
- **Sección 4.** Comparación consolidada de resultados entre las tres etapas.

Cada sección incluye, al final, un bloque de preguntas orientadoras para la discusión en clase.


## Sección 0. Configuración del entorno

Se instalan y cargan las librerías requeridas para las tres etapas del ejercicio. Se emplea un modelo de lenguaje de tamaño reducido con el propósito de que el entrenamiento del adaptador LoRA (Etapa 3) pueda ejecutarse en un entorno con recursos computacionales limitados, como una instancia estándar de Google Colab con GPU T4.


In [24]:
# Instalación de dependencias.
# Se fija un conjunto de versiones compatibles entre sí para evitar conflictos.
!pip install -q -U transformers accelerate peft trl datasets sentence-transformers faiss-cpu pandas


In [25]:
# Importación de librerías.
import random
import textwrap
import numpy as np
import pandas as pd
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer

pd.set_option("display.max_colwidth", None)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo de ejecución: {DEVICE}")


Dispositivo de ejecución: cpu


In [26]:
# Carga del modelo base y del tokenizador.
#
# Se utiliza un modelo instruccional de tamaño reducido (aproximadamente 0.5B
# de parámetros), lo cual permite ejecutar tanto la inferencia como el
# ajuste con LoRA en un entorno con recursos limitados.

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map=DEVICE,
)

print("Modelo cargado correctamente.")


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 350.66it/s]


Modelo cargado correctamente.


In [27]:
# Función auxiliar de generación de texto, común a las tres etapas.
#
# Recibe una lista de mensajes en el formato de plantilla conversacional del
# modelo y devuelve únicamente el texto generado por el asistente.

def generar_respuesta(model, tokenizer, mensajes, max_new_tokens=220, temperature=0.3):
    entrada = tokenizer.apply_chat_template(
        mensajes,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )

    input_ids = entrada["input_ids"].to(model.device)
    attention_mask = entrada["attention_mask"].to(model.device)

    with torch.no_grad():
        salida = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 1e-5),
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )

    texto_generado = tokenizer.decode(
        salida[0][input_ids.shape[1]:], skip_special_tokens=True
    )
    return texto_generado.strip()


In [28]:
# Conjunto fijo de preguntas de evaluación.
#
# Este conjunto se emplea sin modificaciones en las tres etapas del ejercicio,
# con el propósito de que la comparación de resultados sea consistente.

PREGUNTAS_EVALUACION = [
    {
        "id": "Q1",
        "tipo": "Fáctica",
        "pregunta": "¿Hasta qué fecha puedo cancelar una materia sin que quede registrada en el certificado de notas?",
    },
    {
        "id": "Q2",
        "tipo": "Fáctica",
        "pregunta": "¿Cuántos créditos mínimos debe tener una asignatura cursada en otra universidad para poder homologarla?",
    },
    {
        "id": "Q3",
        "tipo": "Ambigua",
        "pregunta": "No alcancé a matricularme a tiempo, ¿qué debo hacer?",
    },
    {
        "id": "Q4",
        "tipo": "Ambigua",
        "pregunta": "Si ya pasó la primera semana de clases y aún no me he matriculado, ¿qué debo hacer?",
    },
    {
        "id": "Q5",
        "tipo": "Fuera de tema",
        "pregunta": "¿Qué películas me recomiendas para el fin de semana?",
    },
]

pd.DataFrame(PREGUNTAS_EVALUACION)


,id,tipo,pregunta
0,Q1,Fáctica,¿Hasta qué fecha puedo cancelar una materia sin que quede registrada en el certificado de notas?
1,Q2,Fáctica,¿Cuántos créditos mínimos debe tener una asignatura cursada en otra universidad para poder homologarla?
2,Q3,Ambigua,"No alcancé a matricularme a tiempo, ¿qué debo hacer?"
3,Q4,Ambigua,"Si ya pasó la primera semana de clases y aún no me he matriculado, ¿qué debo hacer?"
4,Q5,Fuera de tema,¿Qué películas me recomiendas para el fin de semana?


### Reglamento de referencia

El siguiente texto corresponde al contenido íntegro del documento `reglamento_tramites_academicos.docx`, incorporado aquí como una cadena de texto para permitir la ejecución autónoma del notebook. Se recomienda que los estudiantes contrasten manualmente las respuestas generadas en la Etapa 1 contra el contenido de este reglamento, como evidencia directa de las limitaciones del prompt engineering.


In [29]:
REGLAMENTO_TEXTO = """
REGLAMENTO GENERAL DE TRÁMITES ACADÉMICOS
Institución Universitaria Ficticia

TÍTULO I. CANCELACIÓN DE ASIGNATURAS

Artículo 1. Plazo ordinario de cancelación.
El estudiante podrá cancelar una asignatura sin que dicha cancelación quede
registrada en el certificado de calificaciones hasta el día en que se cumpla
el cuarenta por ciento (40%) del calendario académico del periodo. Superado
este plazo, la cancelación solo podrá registrarse hasta el setenta por
ciento (70%) del calendario académico, y en tal caso la asignatura quedará
registrada con la anotación "Cancelada" en el certificado de calificaciones.

Artículo 2. Cancelación extraordinaria por fuerza mayor.
Después del setenta por ciento (70%) del calendario académico, el estudiante
únicamente podrá solicitar la cancelación de una asignatura por motivos de
fuerza mayor debidamente acreditados ante la Secretaría Académica, mediante
solicitud escrita y soporte documental, dentro de los cinco (5) días hábiles
siguientes a la ocurrencia del hecho.

TÍTULO II. HOMOLOGACIÓN DE ASIGNATURAS

Artículo 3. Créditos mínimos para homologación.
Solo podrán homologarse asignaturas cursadas y aprobadas en otra institución
de educación superior cuando el número de créditos académicos de la
asignatura de origen sea igual o superior al ochenta por ciento (80%) de los
créditos de la asignatura equivalente en el plan de estudios vigente.

Artículo 4. Equivalencia de contenidos.
La homologación procederá cuando el Comité de Programa certifique una
equivalencia de contenidos igual o superior al setenta y cinco por ciento
(75%) entre los programas de las asignaturas comparadas. La calificación
mínima aprobatoria de la asignatura de origen para efectos de homologación
es de tres punto cinco (3.5) sobre cinco punto cero (5.0).

TÍTULO III. MATRÍCULA ACADÉMICA

Artículo 5. Periodo ordinario de matrícula.
La matrícula académica ordinaria se realizará dentro de las dos (2) semanas
calendario previas al inicio de clases del respectivo periodo académico, de
acuerdo con el calendario publicado por la Oficina de Admisiones y Registro.

Artículo 6. Matrícula extemporánea.
Vencido el periodo ordinario, el estudiante podrá matricularse de manera
extemporánea únicamente durante la primera semana de clases, previo pago de
un recargo administrativo equivalente al diez por ciento (10%) del valor de
la matrícula. Transcurrida dicha semana, no procederá matrícula alguna para
el periodo en curso.

TÍTULO IV. REQUISITOS DE GRADO

Artículo 7. Créditos y promedio mínimo.
Para optar al título correspondiente, el estudiante deberá haber aprobado la
totalidad de los créditos académicos establecidos en su plan de estudios y
mantener un promedio académico acumulado igual o superior a tres punto cero
(3.0) sobre cinco punto cero (5.0).

Artículo 8. Práctica profesional y trabajo de grado.
El estudiante deberá acreditar la aprobación de la práctica profesional y la
sustentación satisfactoria del trabajo de grado ante un jurado designado por
el Comité de Programa, como requisitos adicionales e independientes para la
obtención del título.

TÍTULO V. DISPOSICIONES GENERALES

Artículo 9. Vigencia.
El presente reglamento rige a partir de su publicación y deroga las
disposiciones que le sean contrarias.

Artículo 10. Canal oficial de atención.
Toda solicitud relacionada con los trámites descritos en este reglamento
deberá presentarse a través de la Oficina de Admisiones y Registro o del
canal digital institucional dispuesto para tal efecto. Este reglamento no
regula asuntos ajenos a los trámites académicos aquí descritos.
"""

print(REGLAMENTO_TEXTO[:400], "...")



REGLAMENTO GENERAL DE TRÁMITES ACADÉMICOS
Institución Universitaria Ficticia

TÍTULO I. CANCELACIÓN DE ASIGNATURAS

Artículo 1. Plazo ordinario de cancelación.
El estudiante podrá cancelar una asignatura sin que dicha cancelación quede
registrada en el certificado de calificaciones hasta el día en que se cumpla
el cuarenta por ciento (40%) del calendario académico del periodo. Superado
este plazo ...


## Sección 1. Etapa 1 — Prompt Engineering

En esta etapa se evalúa el desempeño del modelo base cuando su comportamiento se controla exclusivamente mediante un prompt de sistema, sin proporcionar ningún documento externo como fuente de conocimiento. El prompt define de manera explícita el rol del asistente y el formato exigido para cada respuesta.


In [30]:
SYSTEM_PROMPT = """Eres el Asistente de Trámites Académicos de la institución. \
Tu función es orientar a los estudiantes en consultas relacionadas con \
cancelación de asignaturas, homologaciones, matrícula académica y \
requisitos de grado.

Debes responder siempre siguiendo estrictamente la siguiente estructura, \
en este orden:

Respuesta: [respuesta directa y concisa a la consulta]
Fundamento normativo: [artículo del Reglamento General de Trámites \
Académicos que sustenta la respuesta]
Siguiente paso recomendado: [acción concreta que el estudiante debe \
realizar]

Si la consulta no está relacionada con trámites académicos, debes \
indicarlo con cortesía, sin atender el contenido de dicha consulta, y \
redirigir al estudiante hacia una consulta pertinente."""

print(SYSTEM_PROMPT)


Eres el Asistente de Trámites Académicos de la institución. Tu función es orientar a los estudiantes en consultas relacionadas con cancelación de asignaturas, homologaciones, matrícula académica y requisitos de grado.

Debes responder siempre siguiendo estrictamente la siguiente estructura, en este orden:

Respuesta: [respuesta directa y concisa a la consulta]
Fundamento normativo: [artículo del Reglamento General de Trámites Académicos que sustenta la respuesta]
Siguiente paso recomendado: [acción concreta que el estudiante debe realizar]

Si la consulta no está relacionada con trámites académicos, debes indicarlo con cortesía, sin atender el contenido de dicha consulta, y redirigir al estudiante hacia una consulta pertinente.


In [31]:
# Ejecución de la Etapa 1 sobre el conjunto de preguntas de evaluación.
#
# En esta etapa no se proporciona al modelo ningún fragmento del reglamento;
# el modelo responde exclusivamente a partir de su conocimiento preentrenado
# y de las instrucciones del prompt de sistema.

resultados_etapa1 = []

for item in PREGUNTAS_EVALUACION:
    mensajes = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": item["pregunta"]},
    ]
    respuesta = generar_respuesta(base_model, tokenizer, mensajes)
    resultados_etapa1.append({**item, "respuesta_etapa1": respuesta})
    print(f"[{item['id']}] {item['pregunta']}")
    print(textwrap.indent(respuesta, "    "))
    print("-" * 100)


[Q1] ¿Hasta qué fecha puedo cancelar una materia sin que quede registrada en el certificado de notas?
    Para cancelar una materia sin quedarse registrada en tu certificado de notas, necesitarás presentarte para un examen o prueba previo antes de su fecha límite. Sin embargo, si ya has realizado tus estudios y quieres hacerlo de inmediato, puedes cancelar tu matrícula de manera rápida y eficiente.
----------------------------------------------------------------------------------------------------
[Q2] ¿Cuántos créditos mínimos debe tener una asignatura cursada en otra universidad para poder homologarla?
    Para poder homologar una asignatura cursada en otra universidad, se requiere un mínimo de 20 créditos mínimos.
----------------------------------------------------------------------------------------------------
[Q3] No alcancé a matricularme a tiempo, ¿qué debo hacer?
    Para matricularte a tiempo, te recomiendo seguir estos pasos:

    1. **Comprueba tu solicitud**: Antes de env

### Discusión orientada — Etapa 1

Se recomienda desarrollar esta discusión antes de continuar con la Etapa 2.

1. Contrastar manualmente cada respuesta generada frente al contenido del reglamento (documento `reglamento_tramites_academicos.docx` o la variable `REGLAMENTO_TEXTO`). Identificar los casos en los que el modelo afirma cifras, plazos o porcentajes que no corresponden al documento.
2. Evaluar si el formato de respuesta exigido por el prompt de sistema se cumplió de manera consistente en las cinco preguntas.
3. Formular una conclusión preliminar sobre el alcance del prompt engineering: qué tipo de comportamiento controla de manera efectiva y qué tipo de error no puede prevenir.


## Sección 2. Etapa 2 — RAG (Retrieval-Augmented Generation)

En esta etapa se incorpora un mecanismo de recuperación de información que permite al modelo fundamentar sus respuestas en el contenido real del reglamento. El procedimiento consiste en: segmentar el documento en unidades correspondientes a cada artículo, representar dichas unidades mediante vectores de embeddings, construir un índice de búsqueda por similitud y recuperar, para cada consulta, los fragmentos más relevantes antes de generar la respuesta.


In [32]:
# Segmentación del reglamento en artículos.
#
# Se emplea el encabezado "Artículo" como delimitador. Cada fragmento
# conserva su número y su contenido como una unidad independiente de
# recuperación.

import re

fragmentos_crudos = re.split(r"\n(?=Artículo\s+\d+\.)", REGLAMENTO_TEXTO.strip())
articulos = [frag.strip() for frag in fragmentos_crudos if frag.strip().startswith("Artículo")]

print(f"Número de artículos segmentados: {len(articulos)}")
print("-" * 100)
print(articulos[0])


Número de artículos segmentados: 10
----------------------------------------------------------------------------------------------------
Artículo 1. Plazo ordinario de cancelación.
El estudiante podrá cancelar una asignatura sin que dicha cancelación quede
registrada en el certificado de calificaciones hasta el día en que se cumpla
el cuarenta por ciento (40%) del calendario académico del periodo. Superado
este plazo, la cancelación solo podrá registrarse hasta el setenta por
ciento (70%) del calendario académico, y en tal caso la asignatura quedará
registrada con la anotación "Cancelada" en el certificado de calificaciones.


In [33]:
# Construcción del índice de recuperación.
#
# Se emplea un modelo de embeddings multilingüe, adecuado para trabajar con
# documentos en español, y un índice FAISS basado en similitud de coseno.

from sentence_transformers import SentenceTransformer
import faiss

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
modelo_embeddings = SentenceTransformer(EMBEDDING_MODEL_NAME)

vectores_articulos = modelo_embeddings.encode(
    articulos, normalize_embeddings=True, show_progress_bar=False
)

indice_faiss = faiss.IndexFlatIP(vectores_articulos.shape[1])
indice_faiss.add(np.array(vectores_articulos, dtype="float32"))

print(f"Índice construido con {indice_faiss.ntotal} artículos.")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2784.87it/s]


Índice construido con 10 artículos.


In [34]:
# Función de recuperación de los artículos más relevantes para una consulta.

def recuperar_articulos(pregunta, top_k=2):
    vector_pregunta = modelo_embeddings.encode(
        [pregunta], normalize_embeddings=True, show_progress_bar=False
    )
    distancias, indices = indice_faiss.search(
        np.array(vector_pregunta, dtype="float32"), top_k
    )
    return [articulos[i] for i in indices[0]]


In [17]:
# Función de generación aumentada por recuperación.
#
# El prompt de sistema es el mismo definido en la Etapa 1. Se añade, como
# contexto adicional dentro del mensaje del usuario, el contenido de los
# artículos recuperados para la consulta correspondiente.

def generar_respuesta_rag(model, tokenizer, pregunta, top_k=2, max_new_tokens=220):
    articulos_relevantes = recuperar_articulos(pregunta, top_k=top_k)
    contexto = "\n\n".join(articulos_relevantes)

    mensaje_usuario = (
        f"Artículos del reglamento recuperados como posible contexto:\n\n"
        f"{contexto}\n\n"
        f"Consulta del estudiante: {pregunta}\n\n"
        f"Nota: los artículos anteriores pueden ser o no pertinentes para la "
        f"consulta. Utilízalos únicamente si corresponden."
    )

    mensajes = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": mensaje_usuario},
    ]
    respuesta = generar_respuesta(model, tokenizer, mensajes, max_new_tokens=max_new_tokens)
    return respuesta, articulos_relevantes


In [35]:
# Ejecución de la Etapa 2 sobre el mismo conjunto de preguntas de evaluación.

resultados_etapa2 = []

for item in PREGUNTAS_EVALUACION:
    respuesta, contexto_usado = generar_respuesta_rag(base_model, tokenizer, item["pregunta"])
    resultados_etapa2.append({**item, "respuesta_etapa2": respuesta})
    print(f"[{item['id']}] {item['pregunta']}")
    print("Artículos recuperados:")
    for art in contexto_usado:
        print("   -", art.splitlines()[0])
    print("Respuesta:")
    print(textwrap.indent(respuesta, "    "))
    print("-" * 100)


[Q1] ¿Hasta qué fecha puedo cancelar una materia sin que quede registrada en el certificado de notas?
Artículos recuperados:
   - Artículo 1. Plazo ordinario de cancelación.
   - Artículo 2. Cancelación extraordinaria por fuerza mayor.
Respuesta:
    Respuesta: La cancelación de una materia puede ser realizada hasta el cuarto (4) por ciento (40%) del calendario académico.

    Fundamento normativo: Artículo 1. Plazo ordinario de cancelación.

    Siguiente paso recomendado: Solicita la homologación de la cancelación de la materia antes de hacerla.
----------------------------------------------------------------------------------------------------
[Q2] ¿Cuántos créditos mínimos debe tener una asignatura cursada en otra universidad para poder homologarla?
Artículos recuperados:
   - Artículo 3. Créditos mínimos para homologación.
   - Artículo 7. Créditos y promedio mínimo.
Respuesta:
    Respuesta: El estudiante debe tener un número de créditos académicos de origen igual o superior al o

### Discusión orientada — Etapa 2

1. Verificar si el contenido factual de las respuestas a las preguntas **Q1** y **Q2** corresponde ahora con exactitud al reglamento.
2. Examinar específicamente las respuestas a las preguntas **Q3**, **Q4** y **Q5**, identificando si la estructura de tres partes exigida por el prompt de sistema (Respuesta, Fundamento normativo, Siguiente paso recomendado) se mantuvo, se alteró parcialmente o se omitió.
3. Discutir por qué la incorporación de conocimiento externo mediante RAG no garantiza, por sí sola, la consistencia del comportamiento o del formato del asistente ante consultas ambiguas o ajenas al dominio.
4. A partir de lo anterior, formular la hipótesis de trabajo para la Etapa 3: un comportamiento consistente ante distintos tipos de consulta requiere un ajuste del modelo, y no únicamente instrucciones en el prompt.


## Sección 3. Etapa 3 — LoRA (Low-Rank Adaptation)

En esta etapa se realiza un ajuste de parámetros mediante la técnica LoRA, con el propósito de consolidar el comportamiento y el formato de respuesta exigidos, incluyendo los casos de consultas ambiguas y de consultas fuera del dominio del asistente. El conocimiento factual continúa siendo provisto mediante el mecanismo de recuperación desarrollado en la Etapa 2; el ajuste de parámetros no sustituye a RAG, sino que lo complementa.

El conjunto de datos de entrenamiento se construye con ejemplos que ilustran, de manera explícita, el comportamiento esperado en tres situaciones: consultas fácticas ordinarias, consultas ambiguas y consultas fuera del dominio del asistente.


In [36]:
# Construcción del conjunto de datos de ajuste.
#
# Cada ejemplo especifica la consulta del estudiante y la respuesta esperada,
# ya formateada de acuerdo con la estructura exigida por el prompt de sistema.
# El conjunto incluye variaciones de formulación para favorecer la
# generalización del formato a preguntas no vistas durante el entrenamiento.

EJEMPLOS_ENTRENAMIENTO = [
    # -- Cancelación de asignaturas --------------------------------------
    {
        "pregunta": "¿Cuál es el plazo para cancelar una asignatura sin que quede anotado en mi certificado?",
        "respuesta": (
            "Respuesta: Puede cancelar la asignatura sin anotación en el certificado "
            "hasta que se cumpla el 40% del calendario académico del periodo.\n"
            "Fundamento normativo: Artículo 1 del Reglamento General de Trámites Académicos.\n"
            "Siguiente paso recomendado: Realizar la cancelación a través del sistema de "
            "registro académico antes de la fecha límite correspondiente al 40% del calendario."
        ),
    },
    {
        "pregunta": "Ya pasó el 40% del semestre, ¿todavía puedo cancelar una materia?",
        "respuesta": (
            "Respuesta: Sí, puede cancelarla hasta el 70% del calendario académico, "
            "pero quedará registrada con la anotación \"Cancelada\" en su certificado.\n"
            "Fundamento normativo: Artículo 1 del Reglamento General de Trámites Académicos.\n"
            "Siguiente paso recomendado: Presentar la solicitud de cancelación ante la "
            "Oficina de Admisiones y Registro antes del 70% del calendario académico."
        ),
    },
    {
        "pregunta": "Tuve una emergencia médica después del 70% del semestre, ¿puedo cancelar igual una materia?",
        "respuesta": (
            "Respuesta: Solo mediante cancelación extraordinaria por fuerza mayor, "
            "debidamente acreditada con soporte documental.\n"
            "Fundamento normativo: Artículo 2 del Reglamento General de Trámites Académicos.\n"
            "Siguiente paso recomendado: Radicar la solicitud escrita, junto con el soporte "
            "documental, ante la Secretaría Académica dentro de los 5 días hábiles siguientes al hecho."
        ),
    },
    # -- Homologación de asignaturas --------------------------------------
    {
        "pregunta": "¿Cuántos créditos debe tener una materia de otra universidad para poder homologarla?",
        "respuesta": (
            "Respuesta: La asignatura de origen debe tener un número de créditos igual o "
            "superior al 80% de los créditos de la asignatura equivalente en el plan de estudios vigente.\n"
            "Fundamento normativo: Artículo 3 del Reglamento General de Trámites Académicos.\n"
            "Siguiente paso recomendado: Solicitar al Comité de Programa la revisión de "
            "equivalencia de créditos y contenidos de la asignatura cursada."
        ),
    },
    {
        "pregunta": "¿Con qué nota mínima puedo homologar una asignatura?",
        "respuesta": (
            "Respuesta: La calificación mínima aprobatoria de la asignatura de origen para "
            "efectos de homologación es de 3.5 sobre 5.0.\n"
            "Fundamento normativo: Artículo 4 del Reglamento General de Trámites Académicos.\n"
            "Siguiente paso recomendado: Adjuntar el certificado de calificaciones original en "
            "la solicitud de homologación ante el Comité de Programa."
        ),
    },
    # -- Matrícula académica ------------------------------------------------
    {
        "pregunta": "¿Cuándo es el periodo de matrícula ordinaria?",
        "respuesta": (
            "Respuesta: La matrícula ordinaria se realiza dentro de las dos semanas "
            "calendario previas al inicio de clases del periodo académico.\n"
            "Fundamento normativo: Artículo 5 del Reglamento General de Trámites Académicos.\n"
            "Siguiente paso recomendado: Consultar el calendario académico publicado por la "
            "Oficina de Admisiones y Registro para confirmar las fechas exactas."
        ),
    },
    {
        "pregunta": "No alcancé a matricularme a tiempo, ¿qué debo hacer?",
        "respuesta": (
            "Respuesta: Puede matricularse de manera extemporánea únicamente durante la "
            "primera semana de clases, con un recargo administrativo del 10% del valor de la matrícula.\n"
            "Fundamento normativo: Artículo 6 del Reglamento General de Trámites Académicos.\n"
            "Siguiente paso recomendado: Realizar el pago del recargo y formalizar la matrícula "
            "extemporánea antes de finalizar la primera semana de clases."
        ),
    },
    {
        "pregunta": "Si ya pasó la primera semana de clases y aún no me he matriculado, ¿qué debo hacer?",
        "respuesta": (
            "Respuesta: Transcurrida la primera semana de clases, no procede matrícula alguna "
            "para el periodo académico en curso.\n"
            "Fundamento normativo: Artículo 6 del Reglamento General de Trámites Académicos.\n"
            "Siguiente paso recomendado: Comunicarse con la Oficina de Admisiones y Registro "
            "para conocer las opciones disponibles de cara al siguiente periodo académico."
        ),
    },
    {
        "pregunta": "¿Puedo matricularme tres semanas después de iniciadas las clases?",
        "respuesta": (
            "Respuesta: No. La matrícula extemporánea solo procede durante la primera semana "
            "de clases; transcurrida esta, no procede matrícula alguna para el periodo en curso.\n"
            "Fundamento normativo: Artículo 6 del Reglamento General de Trámites Académicos.\n"
            "Siguiente paso recomendado: Consultar con la Oficina de Admisiones y Registro las "
            "condiciones aplicables para matricularse en el siguiente periodo académico."
        ),
    },
    # -- Requisitos de grado -------------------------------------------------
    {
        "pregunta": "¿Qué promedio necesito para graduarme?",
        "respuesta": (
            "Respuesta: Se requiere un promedio académico acumulado igual o superior a 3.0 "
            "sobre 5.0, además de haber aprobado la totalidad de los créditos del plan de estudios.\n"
            "Fundamento normativo: Artículo 7 del Reglamento General de Trámites Académicos.\n"
            "Siguiente paso recomendado: Verificar su promedio acumulado y los créditos "
            "pendientes ante la Oficina de Admisiones y Registro."
        ),
    },
    {
        "pregunta": "¿La práctica profesional es obligatoria para graduarme?",
        "respuesta": (
            "Respuesta: Sí. Se debe acreditar la aprobación de la práctica profesional y la "
            "sustentación satisfactoria del trabajo de grado, como requisitos independientes.\n"
            "Fundamento normativo: Artículo 8 del Reglamento General de Trámites Académicos.\n"
            "Siguiente paso recomendado: Consultar con el Comité de Programa el estado de "
            "ambos requisitos dentro de su plan de estudios."
        ),
    },
    # -- Consultas fuera de tema (redirección) ------------------------------
    {
        "pregunta": "¿Qué películas me recomiendas para el fin de semana?",
        "respuesta": (
            "Respuesta: Este asistente atiende exclusivamente consultas relacionadas con "
            "trámites académicos.\n"
            "Fundamento normativo: Artículo 10 del Reglamento General de Trámites Académicos.\n"
            "Siguiente paso recomendado: Formular una consulta relacionada con cancelación de "
            "asignaturas, homologaciones, matrícula o requisitos de grado."
        ),
    },
    {
        "pregunta": "¿Cuál es el mejor restaurante cerca del campus?",
        "respuesta": (
            "Respuesta: Este asistente atiende exclusivamente consultas relacionadas con "
            "trámites académicos.\n"
            "Fundamento normativo: Artículo 10 del Reglamento General de Trámites Académicos.\n"
            "Siguiente paso recomendado: Formular una consulta relacionada con cancelación de "
            "asignaturas, homologaciones, matrícula o requisitos de grado."
        ),
    },
    {
        "pregunta": "¿Me puedes ayudar a resolver un ejercicio de cálculo diferencial?",
        "respuesta": (
            "Respuesta: Este asistente atiende exclusivamente consultas relacionadas con "
            "trámites académicos.\n"
            "Fundamento normativo: Artículo 10 del Reglamento General de Trámites Académicos.\n"
            "Siguiente paso recomendado: Formular una consulta relacionada con cancelación de "
            "asignaturas, homologaciones, matrícula o requisitos de grado."
        ),
    },
]

print(f"Número de ejemplos de entrenamiento: {len(EJEMPLOS_ENTRENAMIENTO)}")


Número de ejemplos de entrenamiento: 14


In [37]:
# Formateo del conjunto de datos según la plantilla conversacional del modelo.

from datasets import Dataset

def construir_texto_entrenamiento(pregunta, respuesta):
    mensajes = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": pregunta},
        {"role": "assistant", "content": respuesta},
    ]
    return tokenizer.apply_chat_template(mensajes, tokenize=False)


registros = [
    {"text": construir_texto_entrenamiento(ej["pregunta"], ej["respuesta"])}
    for ej in EJEMPLOS_ENTRENAMIENTO
]

dataset_entrenamiento = Dataset.from_list(registros)
print(dataset_entrenamiento)
print("-" * 100)
print(dataset_entrenamiento[0]["text"])


Dataset({
    features: ['text'],
    num_rows: 14
})
----------------------------------------------------------------------------------------------------
<|im_start|>system
Eres el Asistente de Trámites Académicos de la institución. Tu función es orientar a los estudiantes en consultas relacionadas con cancelación de asignaturas, homologaciones, matrícula académica y requisitos de grado.

Debes responder siempre siguiendo estrictamente la siguiente estructura, en este orden:

Respuesta: [respuesta directa y concisa a la consulta]
Fundamento normativo: [artículo del Reglamento General de Trámites Académicos que sustenta la respuesta]
Siguiente paso recomendado: [acción concreta que el estudiante debe realizar]

Si la consulta no está relacionada con trámites académicos, debes indicarlo con cortesía, sin atender el contenido de dicha consulta, y redirigir al estudiante hacia una consulta pertinente.<|im_end|>
<|im_start|>user
¿Cuál es el plazo para cancelar una asignatura sin que quede 

In [38]:
# Configuración del adaptador LoRA.
#
# Se congelan los parámetros originales del modelo y se introducen matrices
# de bajo rango en las proyecciones de atención, de acuerdo con los valores
# habituales reportados en la literatura para modelos de este tamaño.

from peft import LoraConfig, get_peft_model, TaskType

configuracion_lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)

modelo_lora = get_peft_model(base_model, configuracion_lora)
modelo_lora.print_trainable_parameters()


trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [49]:
# Configuración y ejecución del entrenamiento.
#
# Dado el tamaño reducido del conjunto de datos, se emplea un número
# moderado de épocas. Se recomienda observar la curva de pérdida (loss)
# como indicador de la evolución del ajuste.

from trl import SFTConfig, SFTTrainer

argumentos_entrenamiento = SFTConfig(
    output_dir="./adaptador_lora_tramites",
    num_train_epochs=6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="no",
    report_to=[],
    dataset_text_field="text",
    max_length=512,
    packing=False, use_cpu=(DEVICE == "cpu")
)

entrenador = SFTTrainer(
    model=modelo_lora,
    args=argumentos_entrenamiento,
    train_dataset=dataset_entrenamiento,
    processing_class=tokenizer,
)

resultado_entrenamiento = entrenador.train()


Truncating train dataset: 100%|██████████| 14/14 [00:00<00:00, 4508.27 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 14/14 [00:00<00:00, 6577.83 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
5,2.401078
10,2.221491
15,2.056980
20,1.963232


In [50]:
# Guardado del adaptador entrenado.
#
# Únicamente se almacenan los parámetros del adaptador LoRA, de tamaño
# considerablemente menor al del modelo base, lo cual constituye una de las
# ventajas prácticas de esta técnica.

RUTA_ADAPTADOR = "./adaptador_lora_tramites/adaptador_final"
modelo_lora.save_pretrained(RUTA_ADAPTADOR)
tokenizer.save_pretrained(RUTA_ADAPTADOR)

print(f"Adaptador guardado en: {RUTA_ADAPTADOR}")


Adaptador guardado en: ./adaptador_lora_tramites/adaptador_final


In [51]:
# Función de generación combinada: recuperación (RAG) + modelo ajustado (LoRA).
#
# El mecanismo de recuperación de la Etapa 2 se conserva sin modificaciones;
# el único cambio corresponde al modelo empleado para la generación.

def generar_respuesta_rag_lora(pregunta, top_k=2, max_new_tokens=220):
    return generar_respuesta_rag(
        modelo_lora, tokenizer, pregunta, top_k=top_k, max_new_tokens=max_new_tokens
    )


In [52]:
# Ejecución de la Etapa 3 sobre el mismo conjunto de preguntas de evaluación.

resultados_etapa3 = []

for item in PREGUNTAS_EVALUACION:
    respuesta, contexto_usado = generar_respuesta_rag_lora(item["pregunta"])
    resultados_etapa3.append({**item, "respuesta_etapa3": respuesta})
    print(f"[{item['id']}] {item['pregunta']}")
    print(textwrap.indent(respuesta, "    "))
    print("-" * 100)


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.


[Q1] ¿Hasta qué fecha puedo cancelar una materia sin que quede registrada en el certificado de notas?
    Resp/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API
-------------

### Discusión orientada — Etapa 3

1. Contrastar las respuestas a las preguntas **Q3**, **Q4** y **Q5** con las obtenidas en la Etapa 2. Determinar si el formato exigido se mantuvo de manera consistente en los tres casos.
2. Discutir en qué medida el comportamiento observado corresponde a una generalización del patrón de formato, y no a una memorización literal de los ejemplos de entrenamiento, dado que las preguntas **Q1** y **Q2** no coinciden exactamente con ningún ejemplo del conjunto de datos.
3. Analizar la curva de pérdida (loss) obtenida durante el entrenamiento. Identificar si existen indicios de sobreajuste (loss decreciente de forma sostenida sin estabilización) y discutir qué medidas podrían adoptarse ante dicho escenario (reducción de épocas, ampliación del conjunto de datos, ajuste del rango de LoRA).
4. Formular una conclusión general sobre la complementariedad entre RAG y LoRA: qué tipo de problema resuelve cada técnica y por qué, en este caso, ninguna de las dos resulta suficiente de manera aislada.


## Sección 4. Comparación consolidada de resultados

Se integran en una única tabla las respuestas generadas en las tres etapas para el mismo conjunto de preguntas de evaluación. Esta tabla constituye la evidencia central para la discusión de cierre de la sesión.


In [53]:
# Consolidación de resultados en un único DataFrame.

df_etapa1 = pd.DataFrame(resultados_etapa1)[["id", "tipo", "pregunta", "respuesta_etapa1"]]
df_etapa2 = pd.DataFrame(resultados_etapa2)[["id", "respuesta_etapa2"]]
df_etapa3 = pd.DataFrame(resultados_etapa3)[["id", "respuesta_etapa3"]]

df_comparativo = (
    df_etapa1
    .merge(df_etapa2, on="id")
    .merge(df_etapa3, on="id")
)

df_comparativo


,id,tipo,pregunta,respuesta_etapa1,respuesta_etapa2,respuesta_etapa3
0,Q1,Fáctica,¿Hasta qué fecha puedo cancelar una materia sin que quede registrada en el certificado de notas?,"Para cancelar una materia sin quedarse registrada en tu certificado de notas, necesitarás presentarte para un examen o prueba previo antes de su fecha límite. Sin embargo, si ya has realizado tus estudios y quieres hacerlo de inmediato, puedes cancelar tu matrícula de manera rápida y eficiente.",Respuesta: La cancelación de una materia puede ser realizada hasta el cuarto (4) por ciento (40%) del calendario académico.\n\nFundamento normativo: Artículo 1. Plazo ordinario de cancelación.\n\nSiguiente paso recomendado: Solicita la homologación de la cancelación de la materia antes de hacerla.,Resp/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API
1,Q2,Fáctica,¿Cuántos créditos mínimos debe tener una asignatura cursada en otra universidad para poder homologarla?,"Para poder homologar una asignatura cursada en otra universidad, se requiere un mínimo de 20 créditos mínimos.",Respuesta: El estudiante debe tener un número de créditos académicos de origen igual o superior al ochenta por ciento (80%) de los créditos de la asignatura equivalente en el plan de estudios vigente para poder homologarla.\n\nFundamento normativo: Artículo 3 del Reglamento General de Trámites Académicos\n\nSiguiente paso recomendado: Consulte el artículo 7 del mismo reglamento para obtener más información sobre las condiciones para optar al título correspondiente.,Resp/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API
2,Q3,Ambigua,"No alcancé a matricularme a tiempo, ¿qué debo hacer?","Para matricularte a tiempo, te recomiendo seguir estos pasos:\n\n1. **Comprueba tu solicitud**: Antes de enviar tu solicitud de matrícula, asegúrate de que todos los detalles sean correctos. Pueden ser errores técnicos o falta de información.\n\n2. **Verifica tu número de identificación**: Si has recibido un código de matrícula, asegúrate de tenerlo disponible para la hora de entrega.\n\n3. **Envíe tu solicitud**: Asegúrate de enviar tu solicitud de matrícula antes de las 18 horas de entrega (dependiendo de tu institución). En caso de no cumplir con esta fecha, podrás recibir un rechazo.\n\n4. **Reciba tu matrícula**: Una vez envíate tu solicitud, recibirá tu matrícula por correo elec

In [54]:
# Exportación del resultado consolidado para su revisión y discusión posterior.

RUTA_SALIDA = "comparativo_etapas.csv"
df_comparativo.to_csv(RUTA_SALIDA, index=False)
print(f"Resultado consolidado exportado en: {RUTA_SALIDA}")


Resultado consolidado exportado en: comparativo_etapas.csv


## Cierre de la sesión

Se propone concluir la sesión con la discusión de las siguientes preguntas, a partir de la evidencia recogida en la tabla comparativa:

1. ¿Qué tipo de error observado en la Etapa 1 no pudo corregirse mediante un ajuste del prompt de sistema, y por qué?
2. ¿Qué tipo de error observado en la Etapa 2 no pudo corregirse mediante la incorporación de recuperación de información, y por qué?
3. En un entorno de producción real, ¿bajo qué condiciones sería razonable prescindir de RAG, de LoRA, o de ambas técnicas, y en qué casos resultaría necesario combinarlas?
4. ¿Qué riesgos implica ampliar el conjunto de datos de entrenamiento de la Etapa 3 sin un proceso de revisión y validación de calidad, considerando que dicho conjunto define de manera directa el comportamiento consolidado del asistente?
